## Create dataset with COMETKiwi QE score

In [1]:
import pandas as pd
data_path = "/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/data/raw/divemt/main.tsv"
divemt = pd.read_csv(data_path, sep ='\t')


In [28]:
#create dataframe that only contains pe1 and pe2 in task_type
divemt_filtered = divemt.loc[divemt['task_type'].isin(['pe1','pe2'])].copy()

#prepare inputs for qe model
qe_input = []

for i, (src,mt) in enumerate(zip(divemt_filtered['src_text'], divemt_filtered['mt_text']),start =1):
    input_data = {
        'src': src,
        'mt':mt
    }
    qe_input.append(input_data)


In [33]:
from comet import download_model, load_from_checkpoint
model_path = download_model("Unbabel/wmt22-cometkiwi-da")
cometkiwi = load_from_checkpoint(model_path)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 5295.84it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.2 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-cometkiwi-da/snapshots/1ad785194e391eebc6c53e2d0776cada8f83179a/checkpoints/model.ckpt`
Predicting DataLoader 0:   8%|▊         | 39/516 [16:08<3:17:31, 24.85s/it]
Encoder model frozen.
/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/.venv/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [35]:
model_output = cometkiwi.predict(qe_input, batch_size =16, gpus =0, num_workers =2)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Predicting DataLoader 0: 100%|██████████| 323/323 [1:45:11<00:00, 19.54s/it]


In [36]:
print(len(model_output.scores))

5160


In [37]:
divemt_comet_scored = divemt_filtered.copy()
divemt_comet_scored['qe_cometkiwi'] = model_output.scores

In [38]:
print(divemt_comet_scored.shape)
print(divemt_comet_scored['qe_cometkiwi'].isna().sum())

(5160, 65)
0


In [ ]:
"""
divemt_comet_scored.to_csv(
    "../data/dataset_qe/divemt_cometkiwi_qescore.tsv",
    sep="\t",
    index=False
)
"""